# Day 2 (final) — corpus construction

Every fix from last week's runs, plus two found on review.

| Category | Problem found | Fix |
|---|---|---|
| Induction | Sentence forms (`"The blorp ran fast. The blorp"`) predict `ran`/`was` 20/20 — frame completion, not copying | **Bigram forms only**, both orientations per pair, 24 prompts |
| Induction | Copy check gave 28/32, then 0/32, both wrong | **Prefix match** against nonce words in the prompt; now an `assert` |
| Multi-hop | **5 of 10 cities were their own state capital** — answerable by copying | Non-capital cities only |
| Factual, multi-hop | Base model predicts `" a"` after `"The capital of France is"`; the graph then explains `" a"` | **Answer-mode check**, with and without a `Fact:` prefix |
| Obfuscated | 4 long bases lost rungs 2–3 to length; ladder broken | Base index tracked; **only complete ladders kept**; 4 short bases added |
| Obfuscated | `light_typos` could pick a ≤3-letter word and change nothing | Only words of length >3 are eligible |
| Obfuscated | Entropy non-monotonic (3.56, 3.89, 5.00, 4.20) | Reported, not "fixed" — severity is ordered by construction |
| Syntax | `"...that morning, The keys..."` — capital mid-sentence | Lower-cased |
| ~~MCQ~~| Digit answers (`1776`, `1945`) pushed prompts over 20 tokens | Word-option items |
| ~~MCQ~~| **Letter-based MCQ is unanswerable for Gemma-2-2B base**: arrow form predicts `c` 12/12 (list continuation); answer form picks `a` 75% of the time; constrained accuracy 58% | **Forced choice between two in-context option words**, both orders |~~
| Syntax | Agreement check reported 0% — a column named `head` collided with `DataFrame.head()` | Renamed `head_num`; symmetric logit-margin check |
| All | No per-prompt record of whether the model actually did the task | **`task_ok`** column, saved |
| Entity | 7/8 pairs mismatched on token count | 16 pairs matched **in the frame**, distinct fake surnames |
| Filter | No entropy band kept every category ≥20 | **Filter dropped**; entropy and top-1 kept as covariates |

**Established last week, carried forward:** length eta² = 0.221 overall but **0.081 without MCQ**;
entropy eta² = **0.720** (category determines entropy); entropy–length r = **−0.194**; entity
manipulation 5/16 vs 16/16 round years, **Fisher p = 6.77e-05**.

**Multiple choice dropped** after two failed designs. (see preregistration)

## Part 0 — Install, then restart

Needed only for Part 4. Run alone, then **Run → Restart kernel**.

In [ ]:
!pip install -q transformers accelerate
print("installed — Run > Restart kernel, then continue from Part 1")

## Part 1 — Settings

In [2]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf"
import json, random, re
import numpy as np, pandas as pd

random.seed(0); np.random.seed(0)

MAX_TOKENS = 20
MIN_TOKENS = 5
FACT_PREFIX = "Fact: "        # <-- set to "Fact: " if the Part 5 answer-mode check says so, then rerun

CATEGORIES = ["factual_recall", "arithmetic", "syntactic_agreement", "multi_hop",
              "code_completion", "induction", "entity_known_unknown",
              "obfuscated"]
print(f"MAX_TOKENS={MAX_TOKENS}  FACT_PREFIX={FACT_PREFIX!r}")

MAX_TOKENS=20  FACT_PREFIX='Fact: '


## Part 2 — Generators

In [5]:
# --- 1. FACTUAL RECALL -------------------------------------------------------
CAPITALS = {"France": "Paris", "Japan": "Tokyo", "Italy": "Rome", "Egypt": "Cairo",
            "Brazil": "Brasilia", "Kenya": "Nairobi", "Norway": "Oslo", "Peru": "Lima",
            "Thailand": "Bangkok", "Portugal": "Lisbon", "Greece": "Athens", "Cuba": "Havana"}
SPORTS = {"Michael Jordan": "basketball", "Serena Williams": "tennis",
          "Lionel Messi": "soccer", "Tiger Woods": "golf"}

def gen_factual():
    out = []
    for c_, ans in CAPITALS.items():
        out.append((f"{FACT_PREFIX}The capital of {c_} is", "capital_short", ans))
        out.append((f"They visited the capital of {c_}, which is", "capital_med", ans))
        out.append((f"When travelling abroad last year they visited the capital of {c_}, which is",
                    "capital_long", ans))
    for p, ans in SPORTS.items():
        out.append((f"{FACT_PREFIX}{p} plays the sport of", "sport_short", ans))
    return out


# --- 2. ARITHMETIC -----------------------------------------------------------
PAIRS = [(36, 59), (24, 17), (48, 35), (72, 19), (53, 28), (61, 44),
         (15, 27), (83, 12), (46, 39), (57, 26), (34, 51), (68, 23)]

def gen_arithmetic():
    out = [(f"{a}+{b}=", "bare_short", str(a + b)) for a, b in PAIRS]
    out += [(f"The sum {a}+{b}=", "prefixed_med", str(a + b)) for a, b in PAIRS[:8]]
    out += [(f"Adding the two numbers together we compute that {a}+{b}=", "bare_long", str(a + b))
            for a, b in PAIRS[:8]]
    return out


# --- 3. SYNTACTIC AGREEMENT --------------------------------------------------
SUBJ = ["The keys to the cabinet", "The book on the shelves", "The children in the garden",
        "The author of the novels", "The paintings in the gallery", "The student with the notes"]
FILLS = ["The cat sat on the", "She poured the water into the", "He placed the letter on the",
         "They walked slowly towards the", "The dog ran across the", "I left my jacket in the",
         "We put the boxes under the", "You can find the keys on the"]

def gen_syntax():
    out = []
    for s in SUBJ:
        low = s[0].lower() + s[1:]                     # fixed: no capital mid-sentence
        out.append((s, "np_short", None))
        out.append((f"Earlier that morning, {low}", "np_med", None))
        out.append((f"Despite everything that had happened earlier that morning, {low}",
                    "np_long", None))
    out += [(f, "fill_short", None) for f in FILLS]
    return out


# --- 4. MULTI-HOP ------------------------------------------------------------
# FIXED: every city here is NOT its own state capital. Phoenix, Atlanta, Boston,
# Denver and Nashville were removed - the answer was already in the prompt, so the
# model could copy it and no second hop was required.
STATE_CAP = {"Dallas": "Austin", "Seattle": "Olympia", "Miami": "Tallahassee",
             "Detroit": "Lansing", "Chicago": "Springfield", "Buffalo": "Albany",
             "Pittsburgh": "Harrisburg", "Milwaukee": "Madison", "Memphis": "Nashville",
             "Omaha": "Lincoln"}
CITY_LANG = {"Paris": "French", "Berlin": "German", "Madrid": "Spanish", "Rome": "Italian",
             "Tokyo": "Japanese", "Moscow": "Russian", "Lisbon": "Portuguese"}

def gen_multihop():
    out = []
    for city, ans in STATE_CAP.items():
        out.append((f"{FACT_PREFIX}The capital of the state containing {city} is",
                    "state_capital", ans))
        out.append((f"Thinking about geography, the capital of the state containing {city} is",
                    "state_capital_long", ans))
    for city, ans in CITY_LANG.items():
        out.append((f"A person born in {city} usually speaks", "city_language", ans))
    for city, ans in list(CITY_LANG.items())[:5]:
        out.append((f"Someone who grew up in the city of {city} would normally speak",
                    "city_language_long", ans))
    return out


# --- 5. CODE COMPLETION ------------------------------------------------------
def gen_code():
    snippets = [
        "def add(a, b):\n    return", "def square(x):\n    return x *", "x = 5\ny = x +",
        "print('hello'.", "nums = [3, 1, 2]\nnums.", "s = 'abc'\nprint(s[",
        "def f():\n    pass\n\nf(", "a, b = 1, 2\na, b = b,", "while True:\n    break\nprint(",
        "import os\nos.path.", "data = {}\ndata['key'] =", "assert isinstance(x,",
        "f = lambda x: x *", "counts[k] = counts.get(k,", "text.strip().", "raise ValueError(",
        "x: int =", "for i in range(10):\n    print(", "x = [1, 2, 3]\nprint(len(",
        "import numpy as np\narr = np.", "if x > 0:\n    y = 1\nelse:\n    y =",
        "d = {'a': 1}\nfor k, v in d.", "class Dog:\n    def __init__(self):\n        self.",
        "result = sorted(items, key=lambda x: x.", "with open('f.txt') as f:\n    lines = f.",
        "try:\n    v = int(t)\nexcept ValueError:\n    v =", "total = 0\nfor n in nums:\n    total +=",
        "return [x for x in items if x >", "if not isinstance(v, str):\n    v = str(",
        "np.array([1, 2, 3]).",
    ]
    return [(s, "code", None) for s in snippets]


# --- 6. INDUCTION ------------------------------------------------------------
# Bigram forms ONLY. Sentence frames were solved by frame completion (ran/was) in
# 20/20 cases and did not test induction. Each pair appears in BOTH orientations,
# which doubles the category while keeping every prompt the same length (5 words).
NONCE = ["blorp", "trilve", "zandik", "wemplo", "frugnat", "chelbis",
         "vornik", "plaskew", "drimbo", "quathel", "snerrik", "yolvantz",
         "grelth", "vundix", "phorbal", "tessik", "morvane", "klippet",
         "zarnuk", "brimlow", "havelin", "ruskett", "dolvern", "mirquat"]

def gen_induction():
    out = []
    for a, b in zip(NONCE[::2], NONCE[1::2]):
        out.append((f"{a} {b} {a} {b} {a}", "bigram_ab", b))
        out.append((f"{b} {a} {b} {a} {b}", "bigram_ba", a))
    return out


# --- 7. MULTIPLE CHOICE ------------------------------------------------------
#Removed

# --- 8. KNOWN VS UNKNOWN ENTITY ----------------------------------------------
ENTITY_PAIRS = [
    ("Frida Kahlo", "Frida Batkin"), ("Mary Anning", "Mary Delvano"),
    ("Anne Bronte", "Anne Renhold"), ("Jane Goodall", "Jane Voskell"),
    ("Mary Leakey", "Mary Brithers"), ("Dian Fossey", "Dian Belmore"),
    ("Karen Horney", "Karen Hadric"), ("Kathleen Lonsdale", "Kathleen Corbell"),
    ("Marie Tharp", "Marie Marwick"), ("Anita Brookner", "Anita Denholm"),
    ("Barbara Pym", "Barbara Faskin"), ("Amelia Earhart", "Amelia Tremble"),
    ("Harriet Tubman", "Harriet Ashden"), ("Samuel Pepys", "Samuel Holbeck"),
    ("Robert Hooke", "Robert Dressel"), ("Joseph Priestley", "Joseph Larkhem"),
]

def gen_entity():
    out = []
    for i, (real, fake) in enumerate(ENTITY_PAIRS):
        out.append((f"{real} was born in the year", f"known|p{i:02d}", None))
        out.append((f"{fake} was born in the year", f"unknown|p{i:02d}", None))
    return out


# --- 9. OBFUSCATED -----------------------------------------------------------
# Base index tracked in the subtype so ladder completeness is checkable.
# b0-b7 survived last week intact; b8-b11 are new SHORT bases replacing the four
# long ones whose harsh rungs exceeded MAX_TOKENS.
BASE = ["The weather today is quite", "Please open the front", "She wrote a long",
        "They decided to leave the", "He carried the heavy", "We travelled across the",
        "I forgot to bring my", "The train arrived at the",
        "The children played in the", "My sister bought a new",
        "The river flows past the", "Our teacher asked us to"]

def light_typos(s, rng):
    ws = s.split()
    eligible = [i for i, w in enumerate(ws) if len(w) > 3]   # fixed: never a no-op
    i = rng.choice(eligible)
    w = ws[i]
    j = rng.randrange(1, len(w) - 2)
    ws[i] = w[:j] + w[j + 1] + w[j] + w[j + 2:]
    return " ".join(ws)

def random_caps(s, rng):
    return "".join(ch.upper() if rng.random() < 0.5 else ch.lower() for ch in s)

def char_sub(s, rng):
    sub = {"a": "4", "e": "3", "i": "1", "o": "0", "s": "5", "t": "7"}
    return "".join(sub.get(ch.lower(), ch) if rng.random() < 0.7 else ch for ch in s)

def gen_obfuscated():
    rng = random.Random(1)
    out = []
    for bi, s in enumerate(BASE):
        out.append((s, f"rung0_clean|b{bi:02d}", None))
        out.append((light_typos(s, rng), f"rung1_typos|b{bi:02d}", None))
        out.append((random_caps(s, rng), f"rung2_caps|b{bi:02d}", None))
        out.append((char_sub(s, rng), f"rung3_subst|b{bi:02d}", None))
    return out


GENERATORS = {"factual_recall": gen_factual, "arithmetic": gen_arithmetic,
              "syntactic_agreement": gen_syntax, "multi_hop": gen_multihop,
              "code_completion": gen_code, "induction": gen_induction,
              "entity_known_unknown": gen_entity,
              "obfuscated": gen_obfuscated}

corpus = pd.DataFrame([dict(category=cat, subtype=sub, prompt=txt, expected=exp)
                       for cat, fn in GENERATORS.items() for txt, sub, exp in fn()])

# no rung1 may equal its rung0
ob = corpus[corpus.category == "obfuscated"]
r0 = ob[ob.subtype.str.startswith("rung0")].prompt.values
r1 = ob[ob.subtype.str.startswith("rung1")].prompt.values
assert all(a != b for a, b in zip(r0, r1)), "a typo rung is identical to its clean base"

print(corpus.groupby("category").size().to_string())
print(f"\ntotal drafted: {len(corpus)}")

category
arithmetic              28
code_completion         30
entity_known_unknown    32
factual_recall          40
induction               24
multi_hop               32
obfuscated              48
syntactic_agreement     26

total drafted: 260


## Part 3 — Length, completeness, pairs

CPU only.

In [6]:
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = _tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok

from huggingface_hub import login, whoami
login(token=_tok)
print("logged in as:", whoami()["name"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


logged in as: JSON9776


In [7]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("google/gemma-2-2b")

corpus["n_tok"] = corpus.prompt.apply(lambda p: len(tok(p)["input_ids"]))
over = corpus[~corpus.n_tok.between(MIN_TOKENS, MAX_TOKENS)]
print(f"out of range: {len(over)}")
if len(over):
    print(over[["category", "subtype", "n_tok", "prompt"]].to_string(index=False))
corpus = corpus[corpus.n_tok.between(MIN_TOKENS, MAX_TOKENS)].copy()

# Obfuscated: keep ONLY complete four-rung ladders, so every rung comparison is
# within the same base sentence.
is_ob = corpus.category == "obfuscated"
corpus["base"] = np.where(is_ob, corpus.subtype.str.split("|").str[1], None)
counts = corpus[is_ob].groupby("base").size()
incomplete = set(counts[counts < 4].index)
corpus = corpus[~(is_ob & corpus.base.isin(incomplete))].copy()
print(f"\nobfuscated ladders: {len(counts) - len(incomplete)} complete, "
      f"{len(incomplete)} dropped {sorted(incomplete)}")

# Entity: drop a pair if either member was lost
is_en = corpus.category == "entity_known_unknown"
corpus["pair"] = np.where(is_en, corpus.subtype.str.split("|").str[1], None)
pc = corpus[is_en].groupby("pair").size()
broken = set(pc[pc < 2].index)
corpus = corpus[~(is_en & corpus.pair.isin(broken))].copy()
print(f"entity pairs: {len(pc) - len(broken)} complete, {len(broken)} dropped")

print(f"\nremaining: {len(corpus)}")
print(corpus.groupby("category").size().to_string())

out of range: 2
       category         subtype  n_tok                  prompt
code_completion            code      4       raise ValueError(
     obfuscated rung3_subst|b11     21 0ur 734ch3r 45k3d u5 70

obfuscated ladders: 11 complete, 1 dropped ['b11']
entity pairs: 16 complete, 0 dropped

remaining: 255
category
arithmetic              28
code_completion         29
entity_known_unknown    32
factual_recall          40
induction               24
multi_hop               32
obfuscated              44
syntactic_agreement     26


In [8]:
def eta2(df, val="n_tok", grp="category"):
    gm = df[val].mean()
    sst = ((df[val] - gm) ** 2).sum()
    ssb = sum(len(g) * (g[val].mean() - gm) ** 2 for _, g in df.groupby(grp))
    return ssb / sst

E_ALL = eta2(corpus)
E_NO_MCQ = eta2(corpus[corpus.category != "multiple_choice"])
print(f"eta-squared category -> n_tok: {E_ALL:.3f}   without MCQ: {E_NO_MCQ:.3f}")
print("[last week: 0.221 and 0.081]\n")
print((corpus.groupby("category").n_tok.mean() - corpus.n_tok.mean()).round(1)
      .sort_values().to_string())

eta-squared category -> n_tok: 0.131   without MCQ: 0.131
[last week: 0.221 and 0.081]

category
entity_known_unknown   -1.6
syntactic_agreement    -1.4
obfuscated             -0.8
arithmetic             -0.2
factual_recall          0.2
multi_hop               0.6
code_completion         1.5
induction               2.5


In [9]:
# Entity pairs must match IN THE FRAME: famous surnames get their own token
# precisely because they are frequent, so standalone surname lengths mislead.
def frame_len(name):
    return len(tok(f"{name} was born in the year")["input_ids"])

bad = [(r, f) for r, f in ENTITY_PAIRS if frame_len(r) != frame_len(f)]
fakes = [f.split()[1] for _, f in ENTITY_PAIRS]
print(f"frame-length mismatches: {len(bad)}")
print(f"distinct fake surnames: {len(set(fakes))}/{len(fakes)}")
assert not bad, bad
assert len(set(fakes)) == len(fakes), "a repeated fake surname confounds the unknown arm"

# Induction: nonce words should NOT be single known tokens, or they may be real words.
single = [w for w in NONCE if len(tok(" " + w)["input_ids"]) - 1 <= 1]
print(f"nonce words that are a single token (possibly real words): {single}")

frame-length mismatches: 0
distinct fake surnames: 16/16
nonce words that are a single token (possibly real words): []


## Part 4 — Cheap predictors

Needs GPU. HF model with **`attn_implementation="eager"`** — under the default SDPA attention,
Gemma-2's logit soft-capping is silently disabled. Unbatched, so no padding position can
contaminate the final-token logits.

In [10]:
import torch, gc
from transformers import AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

hf = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b", torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True, attn_implementation="eager",
).cuda().eval()

@torch.no_grad()
def dist(p):
    enc = tok(p, return_tensors="pt").to("cuda")
    return hf(**enc).logits[0, -1].float()

@torch.no_grad()
def measure(prompts):
    P, T, H = [], [], []
    for p in prompts:
        lg = dist(p)
        pr, lp = torch.softmax(lg, -1), torch.log_softmax(lg, -1)
        P.append(pr.max().item())
        T.append(tok.decode([int(pr.argmax())]))
        H.append(float(-(lp.exp() * lp).sum()))
    return P, T, H

corpus["top1_prob"], corpus["top1_token"], corpus["next_token_entropy"] = \
    measure(corpus.prompt.tolist())
corpus["mean_token_logid"] = corpus.prompt.apply(
    lambda p: float(np.mean(np.log1p(tok(p)["input_ids"]))))
print(corpus.groupby("category")[["top1_prob", "next_token_entropy"]].mean().round(3)
      .sort_values("next_token_entropy").to_string())

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

                      top1_prob  next_token_entropy
category                                           
entity_known_unknown      0.828               0.693
induction                 0.883               0.829
code_completion           0.577               1.613
arithmetic                0.649               1.621
factual_recall            0.375               2.807
multi_hop                 0.326               3.162
syntactic_agreement       0.231               3.457
obfuscated                0.174               4.279


In [11]:
# No entropy filter. Last week no band kept every category >= 20 (the gentlest cut
# took MCQ to 18). The distribution is broad with no degenerate tail. Filtering would
# impose different selection pressure on different categories; covariates don't.
print("entropy percentiles [1,5,50,95,99]:",
      np.round(np.percentile(corpus.next_token_entropy, [1, 5, 50, 95, 99]), 2))
E_ENT = eta2(corpus, val="next_token_entropy")
R_ENT_LEN = corpus.next_token_entropy.corr(corpus.n_tok)
print(f"eta-squared category -> entropy: {E_ENT:.3f}   [last week 0.720]")
print(f"r(entropy, n_tok): {R_ENT_LEN:.3f}              [last week -0.194]")
print("\nCategory and entropy are collinear, so they are NOT entered into one model as if")
print("independent. See the pre-registration: models (a) category, (b) cheap continuous")
print("predictors, (c) both with VIF. Model (b) is the practically relevant one.")

entropy percentiles [1,5,50,95,99]: [0.17 0.47 2.78 4.64 5.45]
eta-squared category -> entropy: 0.753   [last week 0.720]
r(entropy, n_tok): -0.109              [last week -0.194]

Category and entropy are collinear, so they are NOT entered into one model as if
independent. See the pre-registration: models (a) category, (b) cheap continuous
predictors, (c) both with VIF. Model (b) is the practically relevant one.


## Part 5 — Category validity

Each category is a claim: *this prompt type engages mechanism M, and attribution handles M badly.*
These checks verify the first half. Without them, a bad score has no interpretation.

In [12]:
# CHECK 1 - ANSWER MODE (factual, multi-hop).
# The attribution graph explains whatever the model predicts. If "The capital of France
# is" predicts " a", the graph explains predicting " a", not capital retrieval.
# circuit-tracer's own demo uses a "Fact:" prefix to push the base model into answer mode.
def answers(pred, expected):
    p = pred.strip().lower()
    return bool(p) and expected.lower().startswith(p)

@torch.no_grad()
def top1(p):
    return tok.decode([int(dist(p).argmax())])

rows = []
for cat in ("factual_recall", "multi_hop"):
    sub = corpus[(corpus.category == cat) & corpus.expected.notna()]
    for _, r in sub.iterrows():
        bare = r.prompt[len(FACT_PREFIX):] if FACT_PREFIX and r.prompt.startswith(FACT_PREFIX) \
               else r.prompt
        rows.append(dict(category=cat, subtype=r.subtype, expected=r.expected,
                         bare=answers(top1(bare), r.expected),
                         fact=answers(top1("Fact: " + bare[0].lower() + bare[1:]), r.expected)))
am = pd.DataFrame(rows)
print(am.groupby(["category", "subtype"])[["bare", "fact"]].mean().round(2).to_string())
print(f"\noverall answer rate: bare {am.bare.mean():.0%}   with 'Fact:' {am.fact.mean():.0%}")
print("\nIf a subtype answers under ~50% bare but well with 'Fact:', set FACT_PREFIX = 'Fact: '")
print("in Part 1 and rerun. The prefix only applies to the *_short / state_capital frames.")

                                   bare  fact
category       subtype                       
factual_recall capital_long        0.75  0.92
               capital_med         0.67  0.92
               capital_short       0.33  0.92
               sport_short         0.75  0.75
multi_hop      city_language       0.86  1.00
               city_language_long  0.80  0.60
               state_capital       0.80  1.00
               state_capital_long  1.00  0.80

overall answer rate: bare 72%   with 'Fact:' 89%

If a subtype answers under ~50% bare but well with 'Fact:', set FACT_PREFIX = 'Fact: '
in Part 1 and rerun. The prefix only applies to the *_short / state_capital frames.


In [13]:
# CHECK 2 - INDUCTION COPYING, by prefix match. Nonce words tokenise into fragments,
# so the first predicted token is e.g. "yol" for "yolvantz". An exact-match check
# reports 0%; a "token appears in prompt" check passes " ran" and reports 88%.
# Both were wrong last week. This is the version that isn't.
NONCE_SET = set(NONCE)
def copies(r):
    pred = r.top1_token.strip().lower()
    if not pred:
        return False
    return r.expected is not None and r.expected.lower().startswith(pred)

ind = corpus[corpus.category == "induction"].copy()
ind["is_copy"] = ind.apply(copies, axis=1)
rate = ind.is_copy.mean()
print(ind.groupby("subtype").is_copy.agg(["sum", "count"]).to_string())
print(f"\ncopying the CORRECT next nonce word: {ind.is_copy.sum()}/{len(ind)} ({rate:.0%})")
print(ind.loc[~ind.is_copy, ["prompt", "expected", "top1_token"]].to_string(index=False))
assert rate >= 0.8, "induction is not eliciting copying - the category does not test induction"


           sum  count
subtype              
bigram_ab   12     12
bigram_ba   12     12

copying the CORRECT next nonce word: 24/24 (100%)
Empty DataFrame
Columns: [prompt, expected, top1_token]
Index: []


In [14]:
# CHECK 3 - OBFUSCATION LADDER. Severity is ordered by construction. Entropy was not
# monotonic last week (rung2 caps > rung3 substitution), plausibly because fully
# substituted text reads as uniformly character-level. Reported, not corrected.
ob = corpus[corpus.category == "obfuscated"].copy()
ob["rung"] = ob.subtype.str.split("|").str[0]
print(ob.groupby("rung")[["next_token_entropy", "top1_prob", "n_tok"]].mean().round(3).to_string())
print(f"\ncomplete ladders: {ob.base.nunique()}  ({len(ob)} prompts)")
print(pd.crosstab(ob.rung, ob.base).to_string())

             next_token_entropy  top1_prob   n_tok
rung                                              
rung0_clean               3.684      0.223   5.636
rung1_typos               4.194      0.170   6.818
rung2_caps                4.963      0.105  11.000
rung3_subst               4.277      0.196  16.000

complete ladders: 11  (44 prompts)
base         b00  b01  b02  b03  b04  b05  b06  b07  b08  b09  b10
rung                                                              
rung0_clean    1    1    1    1    1    1    1    1    1    1    1
rung1_typos    1    1    1    1    1    1    1    1    1    1    1
rung2_caps     1    1    1    1    1    1    1    1    1    1    1
rung3_subst    1    1    1    1    1    1    1    1    1    1    1


In [15]:
# CHECK 4 - ENTITY MANIPULATION. top1_prob does NOT separate the arms: both predict
# a leading space before the year. The year itself does: known names produce
# specific years, unknown names produce round ones.
from scipy.stats import fisher_exact

@torch.no_grad()
def gen(p, n=6):
    enc = tok(p, return_tensors="pt").to("cuda")
    out = hf.generate(**enc, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:])

rows = []
for real, fake in ENTITY_PAIRS:
    for name, kind in ((real, "known"), (fake, "unknown")):
        t = gen(f"{name} was born in the year")
        y = re.search(r"\b(1[0-9]{3})\b", t)
        rows.append(dict(name=name, kind=kind, generated=t.strip(),
                         year=int(y.group(1)) if y else None))
ent = pd.DataFrame(rows)
ent["round"] = ent.year.apply(lambda y: y is not None and y % 5 == 0)
k, u = ent[ent.kind == "known"], ent[ent.kind == "unknown"]
_, P_ENT = fisher_exact([[int(k["round"].sum()), int((~k["round"]).sum())],
                         [int(u["round"].sum()), int((~u["round"]).sum())]])
print(f"round years: known {int(k['round'].sum())}/{len(k)}, unknown {int(u['round'].sum())}/{len(u)}")
print(f"Fisher exact p = {P_ENT:.2e}   [last week 6.77e-05]")
print(f"mean top1_prob: known {corpus[corpus.subtype.str.startswith('known')].top1_prob.mean():.3f}, "
      f"unknown {corpus[corpus.subtype.str.startswith('unknown')].top1_prob.mean():.3f}"
      "   <- confidently wrong, not uncertain")

round years: known 5/16, unknown 16/16
Fisher exact p = 6.77e-05   [last week 6.77e-05]
mean top1_prob: known 0.819, unknown 0.837   <- confidently wrong, not uncertain


In [16]:
# CHECK 5 - READ THEM. No automated check replaces this.
for cat in CATEGORIES:
    sub = corpus[corpus.category == cat]
    print(f"\n=== {cat}  n={len(sub)}  mean {sub.n_tok.mean():.1f} tok ===")
    for _, r in sub.sample(min(4, len(sub)), random_state=0).iterrows():
        print(f"  [{r.n_tok:2d}t H={r.next_token_entropy:4.1f} -> {r.top1_token!r:10s}] {r.prompt!r}")


=== factual_recall  n=40  mean 10.8 tok ===
  [10t H= 3.3 -> ' Lima'   ] 'They visited the capital of Peru, which is'
  [15t H= 3.4 -> ' Oslo'   ] 'When travelling abroad last year they visited the capital of Norway, which is'
  [10t H= 3.4 -> ' Bangkok'] 'They visited the capital of Thailand, which is'
  [10t H= 3.5 -> ' Tokyo'  ] 'They visited the capital of Japan, which is'

=== arithmetic  n=28  mean 10.4 tok ===
  [ 7t H= 1.7 -> '8'       ] '48+35='
  [16t H= 0.5 -> '8'       ] 'Adding the two numbers together we compute that 48+35='
  [10t H= 2.1 -> '8'       ] 'The sum 48+35='
  [10t H= 2.0 -> '1'       ] 'The sum 61+44='

=== syntactic_agreement  n=26  mean 9.2 tok ===
  [15t H= 3.3 -> ' were'   ] 'Despite everything that had happened earlier that morning, the keys to the cabinet'
  [ 7t H= 2.9 -> ' table'  ] 'He placed the letter on the'
  [15t H= 3.1 -> ' were'   ] 'Despite everything that had happened earlier that morning, the paintings in the gallery'
  [15t H= 3.6 -> ' in

In [17]:
ind["frag_len"] = ind.top1_token.str.strip().str.len()
ind["cont"] = ind.prompt.apply(lambda p: gen(p, 3).replace(" ", "").lower())
ind["strict"] = ind.apply(lambda r: r.cont.startswith(r.expected[:3]), axis=1)
print(f"single-character predictions: {(ind.frag_len < 2).sum()}")
print(f"strict copying (first 3 chars over 3 tokens): {ind.strict.sum()}/{len(ind)}")

single-character predictions: 6
strict copying (first 3 chars over 3 tokens): 24/24


In [18]:
# CHECK 7 - SYNTACTIC AGREEMENT. NP prompts pair singular heads with plural distractors
# and vice versa (agreement attraction). Top-1 only tests agreement when the model happens
# to predict a number-marked verb, so the main measure compares plural vs singular verb
# logits directly - symmetric, and gives an effect size.
# NOTE: never name a pandas column "head" - df.head is a method and attribute access
# silently returns it instead of the column. That bug produced a false 0% earlier.
HEAD = {"keys": "pl", "book": "sg", "children": "pl",
        "author": "sg", "paintings": "pl", "student": "sg"}
VERB = {"were": "pl", "are": "pl", "was": "sg", "is": "sg"}
PAIRS_V = [(" were", " was"), (" are", " is")]

syn = corpus[(corpus.category == "syntactic_agreement") &
             corpus.subtype.str.startswith("np")].copy()
syn["head_num"] = syn.prompt.apply(lambda p: next(v for k, v in HEAD.items() if k in p))
syn["verb"] = syn.top1_token.str.strip().map(VERB)

@torch.no_grad()
def plural_margin(p):
    lg = dist(p)
    return float(sum(lg[tok.encode(pl, add_special_tokens=False)[0]] -
                     lg[tok.encode(sg, add_special_tokens=False)[0]] for pl, sg in PAIRS_V))

syn["margin"] = syn.prompt.apply(plural_margin)
syn["agree_ok"] = np.where(syn["margin"] > 0, "pl", "sg") == syn["head_num"]

SYN_TOP1 = (syn["verb"] == syn["head_num"])[syn.verb.notna()].mean()
SYN_AGREE = syn.agree_ok.mean()
print(f"verb predicted: {syn.verb.notna().sum()}/{len(syn)}")
print(f"top-1 agreement where a verb is predicted: {SYN_TOP1:.0%}")
print(syn.groupby("head_num").agg(ok=("agree_ok", "sum"), n=("agree_ok", "size"),
                                 mean_margin=("margin", "mean")).round(2).to_string())
print(f"\nconstrained agreement (symmetric margin): {SYN_AGREE:.0%}")
print("positive margin = plural preferred; want clearly + for pl heads, clearly - for sg heads")
print(syn[["prompt", "head_num", "top1_token", "margin"]].round(2).to_string(index=False))

verb predicted: 10/18
top-1 agreement where a verb is predicted: 100%
          ok  n  mean_margin
head_num                    
pl         9  9        12.23
sg         9  9        -7.51

constrained agreement (symmetric margin): 100%
positive margin = plural preferred; want clearly + for pl heads, clearly - for sg heads
                                                                                 prompt head_num top1_token  margin
                                                                The keys to the cabinet       pl        are    7.88
                                          Earlier that morning, the keys to the cabinet       pl       were   10.88
     Despite everything that had happened earlier that morning, the keys to the cabinet       pl       were   10.38
                                                                The book on the shelves       sg         of   -4.88
                                          Earlier that morning, the book on the shelves       sg  

In [19]:
# TASK SUCCESS, per prompt. Lets Day 6 report every category effect twice: on all
# prompts, and on prompts where the model actually did the task. Left empty where no
# single correct answer exists (code, entity, obfuscated, syntax fill prompts).
corpus["task_ok"] = None
m = corpus.category.isin(["factual_recall", "multi_hop", "arithmetic"]) & corpus.expected.notna()
corpus.loc[m, "task_ok"] = corpus.loc[m].apply(lambda r: answers(r.top1_token, str(r.expected)),
                                               axis=1)
corpus.loc[ind.index, "task_ok"] = ind.strict
corpus.loc[syn.index, "task_ok"] = syn.agree_ok

TASK_OK = {}
for cat, s in corpus.groupby("category").task_ok:
    s = s.dropna()
    TASK_OK[cat] = None if s.empty else float(s.astype(bool).mean())

print(corpus.groupby("category").task_ok.agg(
    defined=lambda s: s.notna().sum(),
    rate=lambda s: s.dropna().astype(bool).mean() if s.notna().any() else float("nan")
).round(2).to_string())

                      defined  rate
category                           
arithmetic                 28  0.96
code_completion             0   NaN
entity_known_unknown        0   NaN
factual_recall             40  0.80
induction                  24  1.00
multi_hop                  32  0.94
obfuscated                  0   NaN
syntactic_agreement        18  1.00


## Part 6 — Save, then download

In [20]:
final = corpus.reset_index(drop=True).copy()
final["prompt_id"] = [f"p{i:04d}" for i in range(len(final))]
final["n_interior"] = final["n_tok"] - 2

print("prompts with fewer than 5 interior positions, by category:")
print(final[final.n_interior < 5].groupby("category").size().to_string())

final = final[["prompt_id", "category", "subtype", "prompt", "expected", "n_tok", "n_interior", "task_ok",
               "top1_prob", "top1_token", "next_token_entropy", "mean_token_logid"]]

os.makedirs("/kaggle/working/out", exist_ok=True)
final.to_csv("/kaggle/working/out/corpus.csv", index=False)
ent.to_csv("/kaggle/working/out/check_entity.csv", index=False)
ind.to_csv("/kaggle/working/out/check_induction.csv", index=False)
am.to_csv("/kaggle/working/out/check_answer_mode.csv", index=False)
syn.to_csv("/kaggle/working/out/check_syntax.csv", index=False)

meta = dict(
    n_prompts=len(final), per_category=final.groupby("category").size().to_dict(),
    max_tokens=MAX_TOKENS, min_tokens=MIN_TOKENS, fact_prefix=FACT_PREFIX,
    filter=None, covariates=["n_tok", "next_token_entropy", "top1_prob", "mean_token_logid"],
    length_eta2=float(E_ALL), length_eta2_no_mcq=float(E_NO_MCQ),
    entropy_eta2=float(E_ENT), r_entropy_ntok=float(R_ENT_LEN),
    induction_copy_rate=float(rate), entity_fisher_p=float(P_ENT),
    answer_rate_bare=float(am.bare.mean()), answer_rate_fact=float(am.fact.mean()),
    obfuscated_ladders=int(ob.base.nunique()),
    n_short_interior=int((final.n_interior < 5).sum()),
    induction_strict_rate=float(ind.strict.mean()),
    syntax_agreement_constrained=float(SYN_AGREE), syntax_agreement_top1=float(SYN_TOP1),
    task_ok_by_category=TASK_OK,
    seed=0,
)
json.dump(meta, open("/kaggle/working/out/corpus_meta.json", "w"), indent=2)
print(json.dumps(meta, indent=2))
print("\n*** DOWNLOAD /kaggle/working/out/ NOW ***")

prompts with fewer than 5 interior positions, by category:
category
code_completion         4
obfuscated             14
syntactic_agreement     9
{
  "n_prompts": 255,
  "per_category": {
    "arithmetic": 28,
    "code_completion": 29,
    "entity_known_unknown": 32,
    "factual_recall": 40,
    "induction": 24,
    "multi_hop": 32,
    "obfuscated": 44,
    "syntactic_agreement": 26
  },
  "max_tokens": 20,
  "min_tokens": 5,
  "fact_prefix": "Fact: ",
  "filter": null,
  "covariates": [
    "n_tok",
    "next_token_entropy",
    "top1_prob",
    "mean_token_logid"
  ],
  "length_eta2": 0.13091481990318063,
  "length_eta2_no_mcq": 0.13091481990318063,
  "entropy_eta2": 0.7531671332568183,
  "r_entropy_ntok": -0.10852979722796455,
  "induction_copy_rate": 1.0,
  "entity_fisher_p": 6.77080814431494e-05,
  "answer_rate_bare": 0.7222222222222222,
  "answer_rate_fact": 0.8888888888888888,
  "obfuscated_ladders": 11,
  "n_short_interior": 27,
  "induction_strict_rate": 1.0,
  "syntax_agre

## Pre-registration entries to add

Paste into the deviations log with today's date, filling in the numbers from `corpus_meta.json`.

**Induction rebuilt.** Sentence-frame prompts were solved by frame completion (`ran`/`was` in
20/20), not copying, and were dropped. The category now uses bigram repetition in both orientations
(24 prompts). Copying is checked by prefix match against the expected next nonce word, since nonce
words tokenise into fragments. Two earlier implementations of this check reported 88% and 0%; both
were wrong, for opposite reasons.

**Multi-hop corrected.** Five of the original ten cities were their own state capital, making the
answer copyable from the prompt. Replaced with non-capital cities.

**Answer mode.** The attribution graph explains whatever the model predicts. Answer rates were
measured with and without a `Fact:` prefix; `FACT_PREFIX` records the choice.

**Obfuscated ladder.** Only complete four-rung ladders retained. Corruption level and token count
are intrinsically confounded, because substituted text fragments into character-level tokens —
which is the mechanism under test. Entropy is not monotonic in corruption severity.

**Entity category.** 16 pairs matched on first name and on token count within the frame. The
manipulation is validated by generated year (round vs specific), not by confidence, which does not
separate the arms. Composition is predominantly women, because less-tokenised surnames correlate
with lesser fame.

**Filter dropped.** No entropy band retained every category above 20. Entropy and top-1 probability
are covariates.

**Collinearity.** Category determines entropy (eta² ≈ 0.72). Pre-specified: three models, with the
cheap-predictors-only model and leave-one-category-out CV as the practically relevant one.

**Length.** Eta² ≈ 0.22 overall, ≈ 0.08 without MCQ, which runs long by construction.

## What the extra budget buys

This week's ~30 GPU hours against ~2 hours for the main run. Options for Day 4, in order of value:

1. **Cross-layer transcoders on the full corpus**, not a 60-prompt subsample. Tests whether hard
   prompts are hard for the *method* or for the *dictionary*. Published gap: 0.61 vs 0.37.
2. **Llama-3.2-1B on the full corpus.** Tests whether findings transfer across models.
3. **A second rater** for the Day 5 human validation.

**Forced choice replaces multiple choice.** ~~Letter-based MCQ failed its manipulation check: Gemma-2-2B
(base) does not perform multiple-choice symbol binding. The arrow format continued the option list
(12/12 predicted `c`); the answer format defaulted to `a` (chosen 75% of the time; accuracy 58%,
against 42% for always answering `a`); constrained choice between the two letters was also 58%. The
category was rebuilt as a forced choice between two option words present in the prompt, in both
orders. Result: constrained [X]%, top-1 [X]%, picks first-listed [X]%. The category is described as
forced choice, not multiple choice.~~
**MCQ has been dropped**

**Syntactic agreement validated.** Where the model predicts a number-marked verb, it agrees with the
true head in [10/10]; by symmetric verb-logit margin, [18/18], with no attraction errors. An earlier
run reported 0% because a pandas column named `head` collided with `DataFrame.head()`.

**Task success recorded.** `task_ok` is stored per prompt wherever a single correct answer exists.
Category effects are reported on all prompts and on the `task_ok` subset.